In [1]:
import re
import time
import unicodedata
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from urllib.parse import unquote, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("جاهز ✓")

pandas: 2.2.3
requests: 2.32.3
جاهز ✓


In [2]:
BASE_URL = "https://en.wikipedia.org"

USER_AGENT = (
    "PremierLeagueResearchBot/1.0 "
    "(personal data-analysis project)"
)

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 30
MAX_RETRIES = 3
RETRY_BACKOFF = 2.0

SEASONS_DEFAULT_START_YEARS = list(range(2010, 2025))

print("المواسم اللي هتتسحب:", SEASONS_DEFAULT_START_YEARS)
print("العدد:", len(SEASONS_DEFAULT_START_YEARS), "موسم")

المواسم اللي هتتسحب: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
العدد: 15 موسم


In [3]:
SEASON_PAGE_SECTIONS = {
    "Stadiums_Locations": ["stadiums and locations", "stadia and locations", "stadiums"],
    "Personnel_Kits":     ["personnel and kits", "team kits and personnel",
                           "personnel and sponsorship"],
    "Managerial_Changes": ["managerial changes", "manager changes"],
    "League_Table":       ["league table", "final league table", "final table"],
    "Results":            ["results"],
    "Top_Scorers":        ["top scorers", "scoring"],
    "Hat_Tricks":         ["hat-tricks", "hat tricks"],
    "Clean_Sheets":       ["clean sheets"],
    "Monthly_Awards":     ["monthly awards", "manager of the month",
                           "player of the month", "awards"],
    "Attendances":        ["attendances", "average attendances", "attendance"],
}

TEAM_PAGE_SECTIONS = {
    "Squad_Information":   ["squad information", "first-team squad",
                            "current squad", "squad", "players"],
    "Transfers_In":        ["transfers in", "in"],
    "Transfers_Out":       ["transfers out", "out"],
    "Loans_Out":           ["loan out", "loans out", "loaned out", "out on loan"],
    "Appearances_Goals":   ["appearances and goals", "appearances",
                            "statistics", "squad statistics"],
    "Disciplinary_Record": ["disciplinary record", "disciplinary"],
    "Coaching_Staff":      ["coaching staff", "current coaching staff", "club staff"],
}

ALL_SHEET_NAMES = (
    ["Seasons", "Teams"]
    + list(SEASON_PAGE_SECTIONS.keys())
    + list(TEAM_PAGE_SECTIONS.keys())
    + ["Scraping_Log"]
)

CITATION_PATTERN = re.compile(
    r"\[\s*\d+\s*\]"
    r"|\[\s*[a-zA-Z]\s*\]"
    r"|\[\s*citation\s+needed\s*\]"
    r"|\[\s*note\s*\d*\s*\]"
    r"|\[\s*nb\s*\d*\s*\]"
    r"|\[\s*update\s*\]"
    r"|\[\s*verification\s+needed\s*\]",
    flags=re.IGNORECASE,
)

print("عدد الـ sheets المتوقعة:", len(ALL_SHEET_NAMES))

عدد الـ sheets المتوقعة: 20


In [4]:
def build_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
    })
    return s


def fetch(session, url):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
            if resp.status_code == 200:
                time.sleep(REQUEST_DELAY_SECONDS)
                return resp.text
            if resp.status_code == 404:
                print(f"  ✗ صفحة مش موجودة: {url}")
                return None
            print(f"  HTTP {resp.status_code} (محاولة {attempt}/{MAX_RETRIES})")
        except requests.RequestException as exc:
            print(f"  خطأ شبكة (محاولة {attempt}/{MAX_RETRIES}): {exc}")
        time.sleep(RETRY_BACKOFF ** attempt)
    return None


print("دوال الإنترنت جاهزة ✓")

دوال الإنترنت جاهزة ✓


In [5]:
def season_label(start_year):
    return f"{start_year}\u2013{str(start_year + 1)[-2:]}"


def season_url(start_year):
    return f"{BASE_URL}/wiki/{season_label(start_year)}_Premier_League"


def normalize_url(href):
    return urljoin(BASE_URL + "/wiki/", href)


print("مثال:", season_url(2023))

مثال: https://en.wikipedia.org/wiki/2023–24_Premier_League


In [6]:
def make_soup(html):
    return BeautifulSoup(html, "lxml")


def heading_text(heading):
    for ed in heading.find_all(class_="mw-editsection"):
        ed.decompose()
    return heading.get_text(" ", strip=True)


def heading_matches(heading, aliases):
    text = heading_text(heading).lower().strip()
    text = re.sub(r"\s+", " ", text)
    for alias in aliases:
        pattern = r"\b" + re.escape(alias.lower().strip()) + r"\b"
        if re.search(pattern, text):
            return True
    return False


def iter_section_nodes(heading):
    level = int(heading.name[1])
    for node in heading.find_all_next():
        if hasattr(node, "name") and node.name and re.match(r"^h[1-6]$", node.name):
            if int(node.name[1]) <= level:
                return
        yield node


def tables_in_section(heading):
    seen = []
    for node in iter_section_nodes(heading):
        if hasattr(node, "name") and node.name == "table":
            classes = node.get("class") or []
            if "wikitable" in classes or "sortable" in classes:
                if node not in seen:
                    seen.append(node)
        elif hasattr(node, "find_all"):
            for t in node.find_all("table", recursive=False):
                classes = t.get("class") or []
                if "wikitable" in classes and t not in seen:
                    seen.append(t)
    return seen


def find_section_tables(soup, aliases):
    for h in soup.find_all(["h2", "h3", "h4"]):
        if heading_matches(h, aliases):
            tabs = tables_in_section(h)
            if tabs:
                return tabs
    return []


print("دوال تحليل HTML جاهزة ✓")

دوال تحليل HTML جاهزة ✓


In [7]:
def standardize_column_name(name):
    s = unicodedata.normalize("NFKC", str(name))
    s = CITATION_PATTERN.sub("", s)
    s = s.replace("\xa0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", " ", s).strip()
    replacements = {
        "P": "Played", "Pld": "Played", "Pts": "Points",
        "GF": "Goals_For", "GA": "Goals_Against", "GD": "Goal_Difference",
        "W": "Won", "D": "Drawn", "L": "Lost",
    }
    return replacements.get(s, s)


def dedupe_columns(df):
    seen = {}
    new_cols = []
    for c in df.columns:
        if c in seen:
            seen[c] += 1
            new_cols.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            new_cols.append(c)
    df.columns = new_cols
    return df


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []
        for tup in df.columns:
            parts = [str(p).strip() for p in tup
                     if p is not None and not str(p).lower().startswith("unnamed")]
            dedup = []
            for p in parts:
                if not dedup or dedup[-1] != p:
                    dedup.append(p)
            new_cols.append(" | ".join(dedup) if dedup else "col")
        df.columns = new_cols
    else:
        df.columns = [str(c).strip() for c in df.columns]
    df.columns = [standardize_column_name(c) for c in df.columns]
    return dedupe_columns(df)


def clean_cells(df):
    def _clean(x):
        if isinstance(x, str):
            x = unicodedata.normalize("NFKC", x)
            x = CITATION_PATTERN.sub("", x)
            x = x.replace("\xa0", " ").replace("\u200b", "")
            x = re.sub(r"\s+", " ", x).strip()
            return x if x != "" else None
        return x
    return df.map(_clean) if hasattr(df, "map") else df.applymap(_clean)


def parse_table(table):
    try:
        dfs = pd.read_html(StringIO(str(table)), flavor="lxml")
    except ValueError:
        return None
    if not dfs:
        return None
    return clean_cells(flatten_columns(dfs[0]))


def add_metadata(df, season, team, source_url, table_name, extracted_at):
    meta = pd.DataFrame({
        "Season":         [season]        * len(df),
        "Team":           [team or ""]    * len(df),
        "Source_URL":     [source_url]    * len(df),
        "Table_Name":     [table_name]    * len(df),
        "Date_Extracted": [extracted_at]  * len(df),
    })
    return pd.concat([meta, df.reset_index(drop=True)], axis=1)


print("دوال معالجة الجداول جاهزة ✓")

دوال معالجة الجداول جاهزة ✓


In [8]:
def team_links_from_season_page(soup, season_lbl):
    prefix = f"/wiki/{season_lbl}_"
    out = {}
    for a in soup.find_all("a", href=True):
        raw_href = a["href"]
        clean_href = raw_href.split("#")[0].split("?")[0]
        decoded_href = unquote(clean_href)
        if not decoded_href.startswith(prefix):
            continue
        if not decoded_href.endswith("_season"):
            continue
        url = normalize_url(clean_href)
        visible = a.get_text(" ", strip=True)
        if visible and len(visible) < 60 and "season" not in visible.lower():
            team = visible
        else:
            slug = decoded_href.split(f"{season_lbl}_", 1)[-1]
            slug = slug.removesuffix("_season")
            team = slug.replace("_", " ").strip()
        team = re.sub(r"\s+", " ", team).strip()
        out.setdefault(team, url)
    return out


print("دالة اكتشاف الفرق جاهزة ✓")

دالة اكتشاف الفرق جاهزة ✓


In [9]:
def extract_sections(soup, sections, season, team, source_url, extracted_at):
    out = {}
    log_rows = []

    for sheet_name, aliases in sections.items():
        tables = find_section_tables(soup, aliases)
        if not tables:
            log_rows.append({
                "Season": season, "Team": team or "",
                "Source_URL": source_url, "Table_Name": sheet_name,
                "Status": "MISSING",
                "Detail": f"مفيش عنوان طابق {aliases!r}",
                "Rows": 0, "Date_Extracted": extracted_at,
            })
            continue

        captured = 0
        for tab in tables:
            df = parse_table(tab)
            if df is None or df.empty:
                continue
            df = add_metadata(df, season, team, source_url, sheet_name, extracted_at)
            out.setdefault(sheet_name, []).append(df)
            captured += len(df)

        log_rows.append({
            "Season": season, "Team": team or "",
            "Source_URL": source_url, "Table_Name": sheet_name,
            "Status": "OK" if captured else "EMPTY",
            "Detail": f"{len(tables)} جدول",
            "Rows": captured, "Date_Extracted": extracted_at,
        })

    return out, log_rows


print("دالة الاستخراج جاهزة ✓")

دالة الاستخراج جاهزة ✓


In [10]:
def process_season(session, start_year, scrape_teams=True):
    season_lbl = season_label(start_year)
    url = season_url(start_year)
    now = datetime.now(timezone.utc).isoformat(timespec="seconds")
    
    all_rows = {}
    all_log = []
    
    def add_to_sheet(sheet, df):
        all_rows.setdefault(sheet, []).append(df)

    print(f"\n[{season_lbl}] بفتح صفحة الموسم...")
    html = fetch(session, url)
    if html is None:
        all_log.append({
            "Season": season_lbl, "Team": "", "Source_URL": url,
            "Table_Name": "(season page)", "Status": "FETCH_FAIL",
            "Detail": "فشل تحميل صفحة الموسم", "Rows": 0,
            "Date_Extracted": now,
        })
        return all_rows, all_log

    soup = make_soup(html)
    add_to_sheet("Seasons", pd.DataFrame([{
        "Season": season_lbl, "Team": "", "Source_URL": url,
        "Table_Name": "Seasons", "Date_Extracted": now,
        "Start_Year": start_year, "End_Year": start_year + 1,
    }]))

    season_tables, season_log = extract_sections(
        soup, SEASON_PAGE_SECTIONS,
        season=season_lbl, team=None, source_url=url, extracted_at=now,
    )
    for sheet, dfs in season_tables.items():
        for d in dfs:
            add_to_sheet(sheet, d)
    all_log.extend(season_log)

    team_map = team_links_from_season_page(soup, season_lbl)
    print(f"[{season_lbl}] لقيت {len(team_map)} فريق")
    if team_map:
        add_to_sheet("Teams", pd.DataFrame([
            {"Season": season_lbl, "Team": t, "Team_URL": u,
             "Source_URL": url, "Table_Name": "Teams", "Date_Extracted": now}
            for t, u in sorted(team_map.items())
        ]))

    if not scrape_teams:
        return all_rows, all_log

    for team, team_url in sorted(team_map.items()):
        print(f"[{season_lbl}]   - {team}")
        team_html = fetch(session, team_url)
        if team_html is None:
            all_log.append({
                "Season": season_lbl, "Team": team, "Source_URL": team_url,
                "Table_Name": "(team page)", "Status": "FETCH_FAIL",
                "Detail": "فشل تحميل صفحة الفريق", "Rows": 0,
                "Date_Extracted": now,
            })
            continue
        team_soup = make_soup(team_html)
        t_tables, t_log = extract_sections(
            team_soup, TEAM_PAGE_SECTIONS,
            season=season_lbl, team=team, source_url=team_url, extracted_at=now,
        )
        for sheet, dfs in t_tables.items():
            for d in dfs:
                add_to_sheet(sheet, d)
        all_log.extend(t_log)

    return all_rows, all_log


print("دالة معالجة الموسم جاهزة ✓")

دالة معالجة الموسم جاهزة ✓


In [11]:
_INVALID_SHEET_CHARS = re.compile(r"[:\\/?*\[\]]")


def _safe_sheet_name(name, used):
    s = _INVALID_SHEET_CHARS.sub("_", name)[:31]
    base = s
    i = 1
    while s in used:
        suffix = f"_{i}"
        s = (base[: 31 - len(suffix)]) + suffix
        i += 1
    used.add(s)
    return s


def write_workbook(sheets, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    used = set()
    ordered = [s for s in ALL_SHEET_NAMES if s in sheets] + \
              [s for s in sheets if s not in ALL_SHEET_NAMES]

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for sheet in ordered:
            df = sheets[sheet]
            if df is None or df.empty:
                df = pd.DataFrame({"_empty": []})
            safe = _safe_sheet_name(sheet, used)
            df.to_excel(writer, sheet_name=safe, index=False)
    print(f"\n✓ اتكتب الملف: {out_path}")
    print(f"✓ عدد الـ sheets: {len(ordered)}")


print("دالة الكتابة جاهزة ✓")

دالة الكتابة جاهزة ✓


In [12]:
SAMPLE_HTML = """
<div>
<p>ده موسم اختباري.</p>
<p>شوف <a href="/wiki/2010%E2%80%9311_Arsenal_F.C._season">Arsenal</a>.</p>

<h2 id="League_table">League table</h2>
<table class="wikitable">
<tr><th>Pos</th><th>Team</th><th>Pld</th><th>Pts</th></tr>
<tr><td>1</td><td>Manchester United</td><td>38</td><td>80</td></tr>
</table>

<h2 id="Top_scorers">Top scorers</h2>
<table class="wikitable">
<tr><th>Rank</th><th>Player</th><th>Goals</th></tr>
<tr><td>1</td><td>Dimitar Berbatov[2]</td><td>20</td></tr>
</table>
</div>
"""

soup = make_soup(SAMPLE_HTML)
season_lbl = season_label(2010)

team_links = team_links_from_season_page(soup, season_lbl)
print("الفرق المُكتشفة:", list(team_links.keys()))
assert "Arsenal" in team_links, "Arsenal مفيش!"

rows, _ = extract_sections(
    soup, SEASON_PAGE_SECTIONS,
    season=season_lbl, team=None,
    source_url="http://test", extracted_at="2026-05-16T00:00:00Z",
)
assert "League_Table" in rows, "League_Table مفيش!"
assert "Top_Scorers" in rows, "Top_Scorers مفيش!"

csv = rows["Top_Scorers"][0].to_csv(index=False)
assert "[2]" not in csv, "علامة [2] مالتنظفتش!"

print("\n✓✓✓ الاختبار نجح. الكود جاهز للـ run الحقيقي.")

الفرق المُكتشفة: ['Arsenal']

✓✓✓ الاختبار نجح. الكود جاهز للـ run الحقيقي.


In [13]:
def team_links_from_season_page(soup, season_lbl):
    """يجيب فرق الدوري الممتاز فقط (20 فريق) من جدول الترتيب،
    مش أي رابط في الصفحة."""
    prefix = f"/wiki/{season_lbl}_"
    out = {}

    # ندوّر على جدول الترتيب (League table) ونجيب الروابط اللي جواه بس
    league_tables = find_section_tables(soup, SEASON_PAGE_SECTIONS["League_Table"])
    search_area = league_tables[0] if league_tables else soup

    for a in search_area.find_all("a", href=True):
        raw_href = a["href"]
        clean_href = raw_href.split("#")[0].split("?")[0]
        decoded_href = unquote(clean_href)
        if not decoded_href.startswith(prefix):
            continue
        if not decoded_href.endswith("_season"):
            continue
        url = normalize_url(clean_href)
        visible = a.get_text(" ", strip=True)
        if visible and len(visible) < 60 and "season" not in visible.lower():
            team = visible
        else:
            slug = decoded_href.split(f"{season_lbl}_", 1)[-1]
            slug = slug.removesuffix("_season")
            team = slug.replace("_", " ").strip()
        team = re.sub(r"\s+", " ", team).strip()
        out.setdefault(team, url)

    return out


print("دالة اكتشاف الفرق المُحدّثة جاهزة ✓ (الدوري الممتاز فقط)")

دالة اكتشاف الفرق المُحدّثة جاهزة ✓ (الدوري الممتاز فقط)


In [14]:
# قائمة فرق الدوري الممتاز المعروفة عبر كل المواسم (للتأكد من الفلترة)
def team_links_from_season_page(soup, season_lbl):
    """يجيب فرق الدوري الممتاز فقط. يبحث في جداول الموسم الرئيسية
    (الترتيب + الملاعب + الأطقم) ويأخذ الروابط المشتركة بينها."""
    prefix = f"/wiki/{season_lbl}_"

    def links_in(area):
        found = {}
        for a in area.find_all("a", href=True):
            clean_href = a["href"].split("#")[0].split("?")[0]
            decoded = unquote(clean_href)
            if not decoded.startswith(prefix):
                continue
            if not decoded.endswith("_season"):
                continue
            visible = a.get_text(" ", strip=True)
            if visible and len(visible) < 60 and "season" not in visible.lower():
                team = visible
            else:
                slug = decoded.split(f"{season_lbl}_", 1)[-1].removesuffix("_season")
                team = slug.replace("_", " ").strip()
            team = re.sub(r"\s+", " ", team).strip()
            if team:
                found.setdefault(team, normalize_url(clean_href))
        return found

    # نجرب الجداول دي بالترتيب، أول واحد يطلّع ما بين 18 و 22 فريق نستخدمه
    for section in ["League_Table", "Stadiums_Locations", "Personnel_Kits"]:
        tables = find_section_tables(soup, SEASON_PAGE_SECTIONS[section])
        if not tables:
            continue
        result = {}
        for tab in tables:
            result.update(links_in(tab))
        if 18 <= len(result) <= 22:
            print(f"   (اكتشفت الفرق من جدول: {section})")
            return result

    # احتياطي: لو مفيش جدول طلّع 20 فريق، نجيب من الصفحة كلها أول 20
    print("   ⚠️ مفيش جدول طلّع 20 فريق بالظبط، بستخدم الصفحة كلها")
    return links_in(soup)


print("دالة اكتشاف الفرق الذكية جاهزة ✓")

دالة اكتشاف الفرق الذكية جاهزة ✓


In [15]:
# كل الفرق اللي لعبت في الدوري الممتاز من 2010 لـ 2025 (مرجع ثابت للفلترة)
PREMIER_LEAGUE_CLUBS = {
    "Arsenal", "Aston Villa", "Barnsley", "Birmingham City", "Blackburn Rovers",
    "Blackpool", "Bolton Wanderers", "Bournemouth", "Brentford",
    "Brighton & Hove Albion", "Brighton and Hove Albion", "Burnley", "Cardiff City",
    "Chelsea", "Crystal Palace", "Everton", "Fulham", "Huddersfield Town",
    "Hull City", "Ipswich Town", "Leeds United", "Leicester City", "Liverpool",
    "Luton Town", "Manchester City", "Manchester United", "Middlesbrough",
    "Newcastle United", "Norwich City", "Nottingham Forest", "Queens Park Rangers",
    "Reading", "Sheffield United", "Southampton", "Stoke City", "Sunderland",
    "Swansea City", "Tottenham Hotspur", "Watford", "West Bromwich Albion",
    "West Ham United", "Wigan Athletic", "Wolverhampton Wanderers",
}


def team_links_from_season_page(soup, season_lbl):
    """يجيب فرق الدوري الممتاز فقط، بالاعتماد على قائمة الفرق المعروفة."""
    prefix = f"/wiki/{season_lbl}_"
    out = {}
    for a in soup.find_all("a", href=True):
        clean_href = a["href"].split("#")[0].split("?")[0]
        decoded = unquote(clean_href)
        if not decoded.startswith(prefix):
            continue
        if not decoded.endswith("_season"):
            continue
        visible = a.get_text(" ", strip=True)
        if visible and len(visible) < 60 and "season" not in visible.lower():
            team = visible
        else:
            slug = decoded.split(f"{season_lbl}_", 1)[-1].removesuffix("_season")
            team = slug.replace("_", " ").strip()
        team = re.sub(r"\s+", " ", team).strip()

        # الفلتر المهم: نقبل الفريق بس لو في قائمة الدوري الممتاز
        if team in PREMIER_LEAGUE_CLUBS:
            out.setdefault(team, normalize_url(clean_href))

    return out


print("دالة اكتشاف الفرق النهائية جاهزة ✓ (بقائمة فرق ثابتة)")

دالة اكتشاف الفرق النهائية جاهزة ✓ (بقائمة فرق ثابتة)


In [16]:
# ============================================================
#  الخلية 13 — الـ RUN الكامل: كل المواسم من 2010 لـ 2024
# ============================================================

# كل المواسم (2010-11 لحد 2024-25)
SEASONS_TO_SCRAPE = SEASONS_DEFAULT_START_YEARS

# True = يجيب صفحات الفرق كمان (اللي إحنا عايزينه)
SCRAPE_TEAM_PAGES = True

# مكان حفظ الملف النهائي
OUTPUT_PATH = r"C:\Users\DELL\Desktop\Projects\Premier League\PremierLeague_2010_2025_Raw.xlsx"

# ------------------------------------------------------------

session = build_session()
all_sheets = {}
all_log = []

for start_year in SEASONS_TO_SCRAPE:
    try:
        rows, log_rows = process_season(
            session, start_year, scrape_teams=SCRAPE_TEAM_PAGES
        )
        for sheet, dfs in rows.items():
            all_sheets.setdefault(sheet, []).extend(dfs)
        all_log.extend(log_rows)
    except Exception as exc:
        print(f"✗ خطأ في موسم {start_year}: {exc}")

# دمج كل sheet مع معالجة الأخطاء
final_sheets = {}
for sheet, dfs in all_sheets.items():
    try:
        final_sheets[sheet] = pd.concat(dfs, ignore_index=True, sort=False)
    except Exception as exc:
        print(f"⚠️ مشكلة في دمج '{sheet}': {exc}")
        fixed = []
        for d in dfs:
            d = d.loc[:, ~d.columns.duplicated()]   # شيل الأعمدة المكررة
            fixed.append(d)
        final_sheets[sheet] = pd.concat(fixed, ignore_index=True, sort=False)
        print(f"   ✓ اتصلّحت '{sheet}'")

final_sheets["Scraping_Log"] = pd.DataFrame(all_log)

write_workbook(final_sheets, OUTPUT_PATH)
print("\n✓✓✓ خلص كل المواسم!")



[2010–11] بفتح صفحة الموسم...
[2010–11] لقيت 42 فريق
[2010–11]   - Arsenal
[2010–11]   - Aston Villa
[2010–11]   - Barnsley
[2010–11]   - Birmingham City
[2010–11]   - Blackburn Rovers
[2010–11]   - Blackpool
[2010–11]   - Bolton Wanderers
[2010–11]   - Bournemouth
[2010–11]   - Brentford
[2010–11]   - Brighton & Hove Albion
[2010–11]   - Burnley
[2010–11]   - Cardiff City
[2010–11]   - Chelsea
[2010–11]   - Crystal Palace
[2010–11]   - Everton
[2010–11]   - Fulham
[2010–11]   - Huddersfield Town
[2010–11]   - Hull City
[2010–11]   - Ipswich Town
[2010–11]   - Leeds United
[2010–11]   - Leicester City
[2010–11]   - Liverpool
[2010–11]   - Luton Town
[2010–11]   - Manchester City
[2010–11]   - Manchester United
[2010–11]   - Middlesbrough
[2010–11]   - Newcastle United
[2010–11]   - Norwich City
[2010–11]   - Nottingham Forest
[2010–11]   - Queens Park Rangers
[2010–11]   - Reading
[2010–11]   - Sheffield United
[2010–11]   - Southampton
[2010–11]   - Stoke City
[2010–11]   - Sunderlan

In [ ]:
import openpyxl

wb = openpyxl.load_workbook(OUTPUT_PATH)
print(f"{'Sheet':<24} {'صفوف':>7}  {'أعمدة':>5}")
print("-" * 50)
for name in wb.sheetnames:
    ws = wb[name]
    print(f"{name:<24} {ws.max_row - 1:>7}  {ws.max_column:>5}")

log_df = pd.read_excel(OUTPUT_PATH, sheet_name="Scraping_Log")
print("\nحالة الأقسام:")
print(log_df["Status"].value_counts().to_string())